# E-Commerce Sales & Customer Analysis

---

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [Business Context](#1.-Business-Context) | Why this analysis matters |
| 2 | [Executive Summary](#2.-Executive-Summary) | Key findings at a glance |
| 3 | [Data Dictionary](#3.-Data-Dictionary) | Column definitions |
| 4 | [Data Loading & Cleaning](#4.-Data-Loading-&-Cleaning) | Import and preprocess data |
| 5 | [Data Quality Checks](#5.-Data-Quality-Checks) | Validate data integrity |
| 6 | [Overview Statistics](#6.-Overview-Statistics) | Key metrics summary |
| 7 | [Sales Analysis](#7.-Sales-Analysis) | Revenue patterns |
| 8 | [Customer Behavior Analysis](#8.-Customer-Behavior-Analysis) | Conversion & engagement |
| 9 | [Customer Segmentation](#9.-Customer-Segmentation) | RFM-based segments |
| 10 | [Discount & Pricing Analysis](#10.-Discount-&-Pricing-Analysis) | Discount impact |
| 11 | [Seasonal & Time Analysis](#11.-Seasonal-&-Time-Analysis) | Time-based patterns |
| 12 | [Correlation Analysis](#12.-Correlation-Analysis) | Feature relationships |
| 13 | [Assumptions & Limitations](#13.-Assumptions-&-Limitations) | Data constraints |
| 14 | [Key Insights & Recommendations](#14.-Key-Insights-&-Recommendations) | Actionable findings |

---

## 1. Business Context

**Objective:** Analyze e-commerce session data to understand customer behavior, sales patterns, and conversion drivers.

**Business Questions:**
- What drives customer purchases?
- Which marketing channels perform best?
- Why do customers abandon carts?
- How do seasons and devices affect sales?

**Expected Outcomes:**
- Identify high-performing channels and categories
- Understand customer segmentation
- Provide actionable recommendations to increase conversions

## 2. Executive Summary

| Metric | Value |
|--------|-------|
| Total Sessions | *To be calculated* |
| Conversion Rate | *To be calculated* |
| Total Revenue | *To be calculated* |
| Avg Order Value | *To be calculated* |
| Cart Abandonment Rate | *To be calculated* |

**Key Findings:**
- *Will be populated after analysis*

## 3. Data Dictionary

| Column | Type | Description |
|--------|------|-------------|
| session_id | string | Unique session identifier |
| customer_id | string | Unique customer identifier |
| visit_date | date | Date of visit (DD-MM-YYYY) |
| visit_month | int | Month of visit (1-12) |
| visit_weekday | int | Day of week (0=Monday, 6=Sunday) |
| visit_season | int | Season (0=Winter, 1=Spring, 2=Summer, 3=Fall) |
| device_type | int | Device (0=Desktop, 1=Mobile, 2=Tablet) |
| user_type | int | User type (0=Returning, 1=New) |
| marketing_channel | int | Channel (0=Organic, 1=Paid Search, 2=Social Media, 3=Email, 4=Direct, 5=Referral) |
| product_category | int | Category (0=Electronics, 1=Clothing, 2=Home & Garden, 3=Books, 4=Sports, 5=Beauty, 6=Toys, 7=Food) |
| unit_price | float | Price per unit |
| quantity | int | Number of items |
| discount_percent | int | Discount percentage applied |
| discount_amount | float | Discount amount in dollars |
| revenue | float | Final revenue after discount |
| pages_viewed | int | Number of pages viewed |
| time_on_site_sec | int | Time spent on site in seconds |
| added_to_cart | int | Whether item was added to cart (0/1) |
| purchased | int | Whether purchase was made (0/1) |
| cart_abandoned | int | Whether cart was abandoned (0/1) |
| payment_method | int | Payment method (0=Credit Card, 1=Debit Card, 2=PayPal, 3=Bank Transfer, 4=Cash on Delivery, 5=Wallet) |
| rating | float | Customer rating (1-5) |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## 4. Data Loading & Cleaning

In [ ]:
try:
    df = pd.read_csv('../data/Ecommerce_sales.csv')
    print(f'Shape: {df.shape}')
    print(f'Missing values: {df.isnull().sum().sum()}')
    print(f'Duplicates: {df.duplicated().sum()}')
    df.head()
except FileNotFoundError:
    print('Error: File not found. Please check the file path.')

### Fix: Map encoded categorical columns to readable labels

In [ ]:
device_map = {0: 'Desktop', 1: 'Mobile', 2: 'Tablet'}
user_map = {0: 'Returning', 1: 'New'}
channel_map = {0: 'Organic', 1: 'Paid Search', 2: 'Social Media', 3: 'Email', 4: 'Direct', 5: 'Referral'}
category_map = {0: 'Electronics', 1: 'Clothing', 2: 'Home & Garden', 3: 'Books', 4: 'Sports', 5: 'Beauty', 6: 'Toys', 7: 'Food'}
payment_map = {0: 'Credit Card', 1: 'Debit Card', 2: 'PayPal', 3: 'Bank Transfer', 4: 'Cash on Delivery', 5: 'Wallet'}
season_map = {0: 'Winter', 1: 'Spring', 2: 'Summer', 3: 'Fall'}

df['device_type_label'] = df['device_type'].map(device_map)
df['user_type_label'] = df['user_type'].map(user_map)
df['channel_label'] = df['marketing_channel'].map(channel_map)
df['category_label'] = df['product_category'].map(category_map)
df['payment_label'] = df['payment_method'].map(payment_map)
df['season_label'] = df['visit_season'].map(season_map)

df['visit_date'] = pd.to_datetime(df['visit_date'], format='%d-%m-%Y')
df['is_purchased'] = df['purchased'].astype(bool)
df['is_cart_abandoned'] = df['cart_abandoned'].astype(bool)

print('Encoded columns mapped successfully.')
df[['device_type_label','user_type_label','channel_label','category_label','payment_label','season_label']].head()

## 5. Data Quality Checks

In [ ]:
print('=== Data Quality Report ===')
print(f'\n1. Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns')

print(f'\n2. Missing Values:')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else '   No missing values')

print(f'\n3. Duplicates: {df.duplicated().sum()} rows')

print(f'\n4. Data Types:')
print(df.dtypes.value_counts())

print(f'\n5. Numeric Column Statistics:')
numeric_cols = ['unit_price', 'quantity', 'revenue', 'pages_viewed', 'time_on_site_sec']
print(df[numeric_cols].describe().round(2))

print(f'\n6. Outlier Detection (IQR method):')
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum()
    print(f'   {col}: {outliers} outliers')

print(f'\n7. Value Counts for Categorical Columns:')
cat_cols = ['device_type_label', 'user_type_label', 'channel_label', 'category_label']
for col in cat_cols:
    print(f'\n   {col}:')
    print(f'   {df[col].value_counts().to_dict()}')

## 6. Overview Statistics

In [ ]:
total_sessions = len(df)
total_purchases = df['purchased'].sum()
conversion_rate = total_purchases / total_sessions * 100
total_revenue = df['revenue'].sum()
avg_order_value = df[df['purchased']==1]['revenue'].mean()
cart_abandon_rate = df['cart_abandoned'].sum() / df['added_to_cart'].sum() * 100

print(f'Total Sessions:       {total_sessions:,}')
print(f'Total Purchases:      {total_purchases:,}')
print(f'Conversion Rate:      {conversion_rate:.1f}%')
print(f'Total Revenue:        ${total_revenue:,.2f}')
print(f'Avg Order Value:      ${avg_order_value:.2f}')
print(f'Cart Abandon Rate:    {cart_abandon_rate:.1f}%')

## 7. Sales Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Revenue by Month
monthly = df[df['purchased']==1].groupby('visit_month')['revenue'].sum()
axes[0,0].bar(monthly.index, monthly.values, color=sns.color_palette('viridis', 12))
axes[0,0].set_title('Revenue by Month')
axes[0,0].set_xlabel('Month')
axes[0,0].set_ylabel('Revenue ($)')

# Revenue by Category
cat_rev = df[df['purchased']==1].groupby('category_label')['revenue'].sum().sort_values(ascending=True)
cat_rev.plot(kind='barh', ax=axes[0,1], color=sns.color_palette('magma', len(cat_rev)))
axes[0,1].set_title('Revenue by Product Category')
axes[0,1].set_xlabel('Revenue ($)')

# Revenue by Device
dev_rev = df[df['purchased']==1].groupby('device_type_label')['revenue'].sum()
axes[1,0].pie(dev_rev, labels=dev_rev.index, autopct='%1.1f%%', colors=sns.color_palette('Set2'))
axes[1,0].set_title('Revenue Share by Device')

# Revenue by Channel
ch_rev = df[df['purchased']==1].groupby('channel_label')['revenue'].sum().sort_values(ascending=True)
ch_rev.plot(kind='barh', ax=axes[1,1], color=sns.color_palette('coolwarm', len(ch_rev)))
axes[1,1].set_title('Revenue by Marketing Channel')
axes[1,1].set_xlabel('Revenue ($)')

plt.tight_layout()
plt.savefig('../reports/sales_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Customer Behavior Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Conversion Rate by Channel
conv_channel = df.groupby('channel_label')['purchased'].mean() * 100
conv_channel.sort_values().plot(kind='barh', ax=axes[0,0], color=sns.color_palette('RdYlGn', len(conv_channel)))
axes[0,0].set_title('Conversion Rate by Channel (%)')
axes[0,0].set_xlabel('Conversion Rate (%)')

# Conversion Rate by Device
conv_device = df.groupby('device_type_label')['purchased'].mean() * 100
conv_device.plot(kind='bar', ax=axes[0,1], color=sns.color_palette('Set2', len(conv_device)))
axes[0,1].set_title('Conversion Rate by Device (%)')
axes[0,1].set_ylabel('Conversion Rate (%)')

# Cart Abandonment Rate by Category
cart_data = df[df['added_to_cart']==1].groupby('category_label')['cart_abandoned'].mean() * 100
cart_data.sort_values(ascending=True).plot(kind='barh', ax=axes[1,0], color=sns.color_palette('Reds_r', len(cart_data)))
axes[1,0].set_title('Cart Abandonment Rate by Category (%)')
axes[1,0].set_xlabel('Abandonment Rate (%)')

# Pages Viewed: Purchased vs Not
df[df['purchased']==1]['pages_viewed'].hist(ax=axes[1,1], alpha=0.6, label='Purchased', bins=20, color='green')
df[df['purchased']==0]['pages_viewed'].hist(ax=axes[1,1], alpha=0.4, label='Not Purchased', bins=20, color='red')
axes[1,1].set_title('Pages Viewed Distribution')
axes[1,1].legend()

plt.tight_layout()
plt.savefig('../reports/customer_behavior.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Customer Segmentation

In [ ]:
customer_df = df.groupby('customer_id').agg(
    total_sessions=('session_id', 'count'),
    total_purchases=('purchased', 'sum'),
    total_revenue=('revenue', 'sum'),
    avg_pages=('pages_viewed', 'mean'),
    avg_time=('time_on_site_sec', 'mean'),
    favorite_category=('category_label', lambda x: x.mode()[0] if len(x.mode()) > 0 else 'Unknown'),
    primary_device=('device_type_label', lambda x: x.mode()[0] if len(x.mode()) > 0 else 'Unknown')
).reset_index()

customer_df['conversion_rate'] = (customer_df['total_purchases'] / customer_df['total_sessions'] * 100).round(1)

# RFM-like segmentation
def segment(row):
    if row['total_purchases'] >= 3 and row['total_revenue'] > 5000:
        return 'Champions'
    elif row['total_purchases'] >= 2 and row['total_revenue'] > 2000:
        return 'Loyal'
    elif row['total_purchases'] >= 1 and row['total_revenue'] > 0:
        return 'Active'
    elif row['total_sessions'] >= 3:
        return 'At Risk'
    else:
        return 'New/Lost'

customer_df['segment'] = customer_df.apply(segment, axis=1)

print('Customer Segments:')
print(customer_df['segment'].value_counts())
print(f'\nTotal Unique Customers: {len(customer_df):,}')
customer_df.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Segment Distribution
seg_counts = customer_df['segment'].value_counts()
seg_counts.plot(kind='bar', ax=axes[0], color=sns.color_palette('viridis', len(seg_counts)))
axes[0].set_title('Customer Segment Distribution')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Revenue by Segment
seg_rev = customer_df.groupby('segment')['total_revenue'].sum().sort_values(ascending=True)
seg_rev.plot(kind='barh', ax=axes[1], color=sns.color_palette('magma', len(seg_rev)))
axes[1].set_title('Total Revenue by Segment')
axes[1].set_xlabel('Revenue ($)')

plt.tight_layout()
plt.savefig('../reports/customer_segments.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Discount & Pricing Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Discount impact on purchase
disc_conv = df.groupby('discount_percent')['purchased'].mean() * 100
disc_conv.plot(kind='bar', ax=axes[0], color=sns.color_palette('RdYlGn', len(disc_conv)))
axes[0].set_title('Conversion Rate by Discount %')
axes[0].set_ylabel('Conversion Rate (%)')
axes[0].set_xlabel('Discount %')

# Avg Revenue by Discount
disc_rev = df[df['purchased']==1].groupby('discount_percent')['revenue'].mean()
disc_rev.plot(kind='bar', ax=axes[1], color=sns.color_palette('YlOrRd', len(disc_rev)))
axes[1].set_title('Avg Order Value by Discount %')
axes[1].set_ylabel('Avg Revenue ($)')
axes[1].set_xlabel('Discount %')

plt.tight_layout()
plt.savefig('../reports/discount_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Seasonal & Time Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Visits by Season
season_visits = df['season_label'].value_counts()
season_visits.plot(kind='bar', ax=axes[0,0], color=sns.color_palette('Set2', len(season_visits)))
axes[0,0].set_title('Sessions by Season')
axes[0,0].tick_params(axis='x', rotation=0)

# Conversion by Season
season_conv = df.groupby('season_label')['purchased'].mean() * 100
season_conv.plot(kind='bar', ax=axes[0,1], color=sns.color_palette('coolwarm', len(season_conv)))
axes[0,1].set_title('Conversion Rate by Season (%)')
axes[0,1].tick_params(axis='x', rotation=0)

# Top Weekdays
weekday_conv = df.groupby('visit_weekday')['purchased'].mean() * 100
weekday_conv.plot(kind='bar', ax=axes[1,0], color=sns.color_palette('viridis', 7))
axes[1,0].set_title('Conversion Rate by Day of Week')

# Revenue Trend by Month
monthly_trend = df[df['purchased']==1].groupby('visit_month').agg(
    revenue=('revenue', 'sum'),
    orders=('purchased', 'sum')
)
ax2 = axes[1,1].twinx()
axes[1,1].bar(monthly_trend.index, monthly_trend['revenue'], color='steelblue', alpha=0.7, label='Revenue')
ax2.plot(monthly_trend.index, monthly_trend['orders'], color='red', marker='o', label='Orders')
axes[1,1].set_title('Revenue & Orders Trend')
axes[1,1].set_xlabel('Month')
axes[1,1].set_ylabel('Revenue ($)')
ax2.set_ylabel('Orders')

plt.tight_layout()
plt.savefig('../reports/seasonal_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Correlation Analysis

In [ ]:
num_cols = ['unit_price','quantity','discount_percent','discount_amount','revenue',
            'pages_viewed','time_on_site_sec','added_to_cart','purchased','cart_abandoned','rating']

fig, ax = plt.subplots(figsize=(12, 10))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, square=True)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('../reports/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Assumptions & Limitations

**Assumptions:**
- Each row represents a unique session
- Customer IDs are consistent across sessions
- Revenue is calculated after discounts
- Ratings are provided only for purchases

**Limitations:**
- No temporal data beyond month/day (no hour/minute)
- No geographic data (location, region)
- No marketing spend data (only channel)
- No competitor pricing data
- Cart abandonment may include multi-item carts

## 14. Key Insights & Recommendations

In [ ]:
insights = f"""
{'='*60}
KEY INSIGHTS & RECOMMENDATIONS
{'='*60}

1. CONVERSION
   - Overall conversion rate: {conversion_rate:.1f}%
   - Best channel: {conv_channel.idxmax()} ({conv_channel.max():.1f}%)
   - Worst channel: {conv_channel.idxmin()} ({conv_channel.min():.1f}%)

2. CART ABANDONMENT
   - Abandonment rate: {cart_abandon_rate:.1f}%
   - Biggest issue category: {cart_data.idxmax()} ({cart_data.max():.1f}%)

3. REVENUE
   - Total revenue: ${total_revenue:,.2f}
   - Average order value: ${avg_order_value:.2f}
   - Top category: {cat_rev.idxmax()} (${cat_rev.max():,.2f})

4. DEVICE
   - Desktop revenue share: {dev_rev.get('Desktop', 0)/dev_rev.sum()*100:.1f}%
   - Mobile conversion: {conv_device.get('Mobile', 0):.1f}%

5. RECOMMENDATIONS
   - Focus on {conv_channel.idxmin()} channel optimization
   - Reduce cart abandonment in {cart_data.idxmax()} category
   - Target {seg_counts.idxmax()} segment for upselling
   - Leverage {season_visits.idxmax()} season campaigns
{'='*60}
"""
print(insights)

In [ ]:
df.to_csv('../data/Ecommerce_sales_cleaned.csv', index=False)
customer_df.to_csv('../reports/customer_segments.csv', index=False)
print('Cleaned data saved to data/Ecommerce_sales_cleaned.csv')
print('Customer segments saved to reports/customer_segments.csv')